# Growth Gap Analysis

## Data Import and Libraries

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import os
from pathlib import Path
from dotenv import load_dotenv,find_dotenv

from sqlalchemy import create_engine, text


import warnings
warnings.filterwarnings('ignore')

In [ ]:
env_path =Path.cwd()/"github_repo_cloned"/"india-logistics-hub-optimizer"/".env"
load_dotenv(env_path) 

HOST = os.getenv("DB_HOST")
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
OUTPUT_DIR = os.getenv("OUTPUT_DIR")

In [3]:
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}")


def sql(query, engine = engine):
    with engine.connect() as conn:
        df = pd.read_sql(text(query),conn)
    return df
     

In [47]:
query = 'select * from gsdp_growth'
gsdp = sql(query=query, engine = engine)

query = 'select * from bill_growth'
bill = sql(query = query, engine = engine)

In [48]:
gsdp.head()

,state,19-20,20-21,21-22,22-23
0,andhra pradesh,5.97,5.70,17.36,13.50
1,arunachal pradesh,18.51,1.67,13.92,13.96
2,assam,12.13,-2.03,21.09,19.86
3,bihar,10.20,-2.51,14.64,15.55
4,chhattisgarh,5.37,2.22,16.52,13.12


In [49]:
bill.head()

,state,19-20,20-21,21-22,22-23
0,jammu and kashmir,89.54,-0.50,31.49,20.96
1,himachal pradesh,100.47,-7.39,31.71,15.29
2,punjab,92.89,-5.74,39.33,18.82
3,chandigarh,93.95,-14.89,35.10,17.37
4,uttarakhand,95.29,-12.67,26.41,20.08


## Initial Data Check

In [50]:
print("Any Duplicate Values?")
print(f"GSDP :{gsdp['state'].duplicated().sum()}")
print(f"Bill: {bill['state'].duplicated().sum()}")


Any Duplicate Values?
GSDP :0
Bill: 0


In [51]:
print("Any Null Values?")
print(f"GSDP :\n{gsdp.isnull().sum()}")
print('-'*30)
print(f"Bill: \n{bill.isnull().sum()}")


Any Null Values?
GSDP :
state    0
19-20    0
20-21    0
21-22    0
22-23    0
dtype: int64
------------------------------
Bill: 
state    0
19-20    1
20-21    1
21-22    0
22-23    0
dtype: int64


In [52]:
bill[bill.isnull().any(axis=1)]

,state,19-20,20-21,21-22,22-23
34,ladakh,NaN,NaN,80.59,55.28


```Ladakh will be removed, as 2 years of ladakh have missing values, the result might not be good enough```

In [53]:
bill = bill[~bill['state'].isin(['ladakh'])]
gsdp = gsdp[~gsdp['state'].isin(['ladakh'])]

In [55]:
bill.info()

<class 'pandas.core.frame.DataFrame'>
Index: 34 entries, 0 to 33
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   state   34 non-null     object 
 1   19-20   34 non-null     float64
 2   20-21   34 non-null     float64
 3   21-22   34 non-null     float64
 4   22-23   34 non-null     float64
dtypes: float64(4), object(1)
memory usage: 1.6+ KB


In [56]:
gsdp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   state   33 non-null     object 
 1   19-20   33 non-null     float64
 2   20-21   33 non-null     float64
 3   21-22   33 non-null     float64
 4   22-23   33 non-null     float64
dtypes: float64(4), object(1)
memory usage: 1.4+ KB


In [59]:
gsdp.describe(include='number').round(2)

,19-20,20-21,21-22,22-23
count,33.00,33.00,33.00,33.00
mean,8.01,-1.20,17.02,13.77
std,3.47,3.32,3.73,3.20
min,1.51,-9.22,4.04,1.20
25%,5.60,-3.24,14.83,13.25
50%,8.06,-1.17,16.95,14.05
75%,9.71,1.59,19.34,15.49
max,18.51,5.70,22.90,19.86


## Bills Growth

## GSDP Growth

## Comparison